# 🔬 Visualizing Electron Density in Proteins

---

## Learning Objectives

By the end of this lesson, you will be able to:
- Understand what electron density maps represent
- Work with 2Fo-Fc and Fo-Fc difference maps
- Use Jmol/PyMOL to visualize electron density
- Interpret density quality and fit

---

## What is Electron Density?

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    X-RAY CRYSTALLOGRAPHY WORKFLOW                       │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Protein Crystal  →  X-ray Beam  →  Diffraction Pattern  →  Model     │
│        │                   │                │                  │        │
│    ┌───┴───┐          ┌───┴───┐        ┌───┴───┐          ┌───┴───┐    │
│    │ ::::: │    →     │   ◉   │   →    │ spots │    →     │ atoms │    │
│    │ ::::: │          │  / \  │        │ array │          │ coord │    │
│    └───────┘          └───────┘        └───────┘          └───────┘    │
│                                                                         │
│   The diffraction pattern contains AMPLITUDES but NOT PHASES            │
│   → This is the "Phase Problem" in crystallography                      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

**Electron density** is a 3D map showing where electrons are located in the crystal structure. It's calculated from diffraction data and used to build atomic models.

---

## Types of Electron Density Maps

### 1. 2Fo-Fc Map (Blue mesh)
- Shows the overall electron density
- Contoured at 1.0-1.5 σ (sigma)
- Model should fit inside this density

### 2. Fo-Fc Difference Map (Green/Red)
- **Green (+)**: Missing atoms (need to add)
- **Red (-)**: Extra atoms (need to remove)
- Contoured at ±3.0 σ

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    ELECTRON DENSITY INTERPRETATION                      │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│    GOOD FIT:              MISSING ATOMS:           WRONG ATOMS:         │
│                                                                         │
│    ╭───────╮              ╭───────╮                ╭───────╮            │
│   ╱  ───○   ╲            ╱   ○    ╲              ╱    ○    ╲           │
│  │  ─○─ ─○─  │          │  GREEN   │            │    RED   │           │
│   ╲   ○───  ╱            ╲  BLOB   ╱              ╲  BLOB  ╱            │
│    ╰───────╯              ╰───────╯                ╰───────╯            │
│                                                                         │
│    Atoms fit in          Positive Fo-Fc           Negative Fo-Fc       │
│    2Fo-Fc density        = add atoms here         = remove/move atoms  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## Resolution and Density Quality

| Resolution | Features Visible | Typical Use |
|------------|-----------------|-------------|
| < 1.0 Å | Individual atoms | High-quality drug design |
| 1.0-1.5 Å | Most atoms clearly | Detailed mechanism studies |
| 1.5-2.5 Å | Side chains visible | Standard structural biology |
| 2.5-3.5 Å | Main chain, some side chains | Domain/fold analysis |
| > 3.5 Å | Secondary structure only | Large complexes |

```
    1.0 Å              2.0 Å              3.0 Å              4.0 Å
    ┌────┐             ┌────┐             ┌────┐             ┌────┐
    │○ ○ │             │ ○○ │             │ ○  │             │    │
    │ ○○ │             │ ○  │             │○   │             │ ○  │
    │○  ○│             │  ○ │             │    │             │    │
    └────┘             └────┘             └────┘             └────┘
    Atomic             Atoms              Blobs              Unclear
    detail             merged             only               shape
```

---

## Jmol Commands for Electron Density

```javascript
// Load structure with electron density from EDS server
load =1abc

// Load 2Fo-Fc map (blue)
isosurface surf2fofc color lightblue sigma 1.5 
  "http://eds.bmc.uu.se/eds/dfs/ab/1abc/1abc.omap"

// Load Fo-Fc difference map (green/red)
isosurface surffofc+ color green sigma 3.0 
  "http://eds.bmc.uu.se/eds/dfs/ab/1abc/1abc_diff.omap"
isosurface surffofc- color red sigma -3.0 
  "http://eds.bmc.uu.se/eds/dfs/ab/1abc/1abc_diff.omap"

// Focus on specific residue
select :42
center selected
```

In [ ]:
# Python: Download electron density from EDS
import urllib.request

def download_electron_density(pdb_id, output_dir="."):
    """
    Download electron density maps from Uppsala EDS server.
    
    Parameters:
        pdb_id: 4-letter PDB code
        output_dir: Directory to save files
    """
    pdb_id = pdb_id.lower()
    middle = pdb_id[1:3]  # e.g., 'ab' from '1abc'
    
    base_url = f"http://eds.bmc.uu.se/eds/dfs/{middle}/{pdb_id}"
    
    files = {
        f"{pdb_id}.omap": "2Fo-Fc map",
        f"{pdb_id}_diff.omap": "Fo-Fc difference map"
    }
    
    for filename, description in files.items():
        url = f"{base_url}/{filename}"
        output_path = f"{output_dir}/{filename}"
        print(f"Downloading {description}: {url}")
        try:
            urllib.request.urlretrieve(url, output_path)
            print(f"  Saved to {output_path}")
        except Exception as e:
            print(f"  Error: {e}")

# Example usage (uncomment to run)
# download_electron_density("1crn")

---

## Quality Metrics

### R-factor and R-free

| Metric | Good | Acceptable | Poor |
|--------|------|------------|------|
| R-factor | < 0.20 | 0.20-0.25 | > 0.25 |
| R-free | < 0.25 | 0.25-0.30 | > 0.30 |
| R-free - R-factor | < 0.05 | 0.05-0.07 | > 0.07 |

```
           Observed Structure Factors (Fo)
                      vs
           Calculated Structure Factors (Fc)
                      ↓
              R = Σ|Fo - Fc| / Σ|Fo|
                      ↓
         Lower R-factor = Better fit to data
```

---

## 🏋️ Exercises

### Exercise 1: Explore Crambin (1CRN)
Load PDB 1CRN in Jmol and examine the electron density at different sigma levels.

### Exercise 2: Find Problem Areas
Look at the Fo-Fc map to identify regions that might need model correction.

### Exercise 3: Compare Resolutions
Compare electron density of a high-resolution structure (< 1.5 Å) with a lower resolution one (> 3.0 Å).

---

## 📚 Resources

- [Uppsala EDS Server](http://eds.bmc.uu.se/eds/)
- [PDB Validation Reports](https://www.rcsb.org/docs/general-help/validation-reports)
- [Jmol Documentation](http://jmol.sourceforge.net/docs/)

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/07_Kodomo_Structural_Bioinformatics/01_Electron_Density/01_Visualizing_Electron_Density.ipynb`)
